# Practice 4 · Tacotron in Code

**In class** · Goal: run your first code on HaiMa and learn how to interrupt it.

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |


## Question 1 · CBHG Highway Network

CBHG (Convolutional Bank, Highway Network, and GRU) is a powerful feature extraction module originally introduced in the Tacotron text-to-speech architecture. It is designed to capture both local context and long-term sequential dependencies from input data.By combining the local feature extraction of CNNs with the sequential modeling of bidirectional RNNs, CBHG effectively transforms input sequences into rich, context-aware representations.

The CBHG module in Tacotron uses Highway Networks to control information flow. 
The forward pass of a Highway Network is defined as:

$$
g = \sigma(x W_g + b_g)
$$
$$
t = \text{ReLU}(x W_t + b_t)
$$
$$
y = g \odot t + (1 - g) \odot x
$$

Implement `highway_forward(x, W_gate, b_gate, W_transform, b_transform)` using NumPy.
- `x`: Input tensor of shape `(N, D)`.
- `W_gate`, `b_gate`, `W_transform`, `b_transform`: Weight and bias parameters.
- Return the output tensor of shape `(N, D)`.

**Task**: Fill in the missing blocks to compute the gate, transform, and final output.


In [ ]:

# Grid Type: Autograded answer
import numpy as np

def highway_forward(x, W_gate, b_gate, W_transform, b_transform):
    # 🟢 EDIT: Compute the gate and transform
    # YOUR CODE HERE
    gate_logits = x @ W_gate + b_gate
    # Numerically stable sigmoid.
    gate = np.exp(-np.logaddexp(0.0, -gate_logits))
    transform = np.maximum(0.0, x @ W_transform + b_transform)
    
    # 🟢 EDIT: Compute the final highway output
    # YOUR CODE HERE
    out = gate * transform + (1.0 - gate) * x
    
    return out
    

In [ ]:

# Grid Type: Autograded tests.



## Question 2 · Tacotron Decoder Stop Token Target

The Tacotron decoder generates \( r \) frames at each step and predicts a stop token to terminate the sequence. During training, we need to prepare the target sequence for the stop token prediction.

Implement `generate_stop_targets(T, r)`:
- `T`: Total number of frames in the ground truth spectrogram.
- `r`: Number of frames generated per decoder step (reduction factor).
- Calculate the total number of decoder steps: `num_steps = ceil(T / r)`.
- Return a 1D NumPy array of length `num_steps`. All elements should be `0`, except the last element which should be `1` (indicating the stop token).

**Task**: Fill in the missing blocks to calculate the number of steps and generate the target array.


In [ ]:

# Grid Type: Autograded answer
import numpy as np

def generate_stop_targets(T, r):
    # 🟢 EDIT: Calculate the number of decoder steps
    # YOUR CODE HERE
    num_steps = int(np.ceil(T / r))
    
    # 🟢 EDIT: Create the target array and set the last element to 1
    # YOUR CODE HERE
    targets = np.zeros(num_steps, dtype=float)
    if num_steps > 0:
        targets[-1] = 1.0
    
    return targets


In [ ]:

# Grid Type: Autograded tests. Points: 10



## Question 3 · Tacotron Target Generation with Padding Mask

Tacotron is an end-to-end text-to-speech model:
- The decoder generates `r` spectrogram frames at each decoding step.
- The post-processing module refines the decoder-generated spectrogram.
- Training uses two losses (calculated via Mean Squared Error, i.e., the global mean of squared differences across all elements):
  1. The difference between the decoder output and the target spectrogram.
  2. The difference between the post-processed output and the target spectrogram.



The Tacotron decoder generates $r$ frames at each time step and predicts an "End?" (stop) token. During batch training, sequences have different lengths, so we must generate target Stop tokens and a Mask to ignore padding regions when calculating the loss.

Implement `generate_tacotron_targets(target_lengths, max_T, r)`:
- `target_lengths`: A 1D NumPy array of the true frame lengths for each sequence in the batch.
- `max_T`: The maximum frame length in the batch (padded length).
- `r`: The reduction factor (frames generated per decoder step).

Return a tuple `(stops, masks)`:
1. `stops`: Shape `(batch_size, num_steps)`. For each sequence, the step corresponding to its last valid frame should be `1.0`, and all others `0.0`.
2. `masks`: Shape `(batch_size, num_steps)`. For each sequence, the steps covering valid frames (up to its true length) should be `1.0`, and padding steps should be `0.0`.

**Task**: Fill in the missing blocks to initialize the arrays, calculate actual steps, and set the targets.


In [ ]:

# Grid Type: Autograded answer
import numpy as np

def generate_tacotron_targets(target_lengths, max_T, r):
    batch_size = len(target_lengths)
    num_steps = int(np.ceil(max_T / r))
    
    # 🟢 EDIT: Initialize the stops and masks arrays with zeros
    # YOUR CODE HERE
    stops = np.zeros((batch_size, num_steps), dtype=float)
    masks = np.zeros((batch_size, num_steps), dtype=float)
    
    for i, length in enumerate(target_lengths):
        # 🟢 EDIT: Calculate the actual decoder steps for this sequence based on its length
        # YOUR CODE HERE
        actual_steps = int(np.ceil(length / r))
        
        # 🟢 EDIT: Set the stop token for the last step and fill the mask for valid steps
        # YOUR CODE HERE
        if actual_steps > 0:
            stops[i, actual_steps - 1] = 1.0
        masks[i, :actual_steps] = 1.0
        
    return stops, masks
    

In [ ]:

# Grid Type: Autograded tests.
